# Quantum SPN Cryptanalysis — Interactive Walkthrough

This notebook walks the pipeline one stage at a time, so you can see each piece
work before it is composed into the full study:

1. The classical cipher
2. Why one plaintext/ciphertext pair is not enough
3. The reversible encryption circuit — and proving it matches the cipher
4. The oracle, and checking that it uncomputes
5. Grover amplification and over-rotation
6. Noise
7. Readout-error mitigation

Run `pytest -q` first if you have not; these same properties are asserted there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qspn.spn import (
    SPNParams, SBOX, encrypt, decrypt, brute_force_keys,
    make_attack_instance, required_pairs,
)
from qspn.oracle import encryption_circuit, key_search_oracle
from qspn.grover import grover_circuit, optimal_iterations, success_probability
from qspn.noise import NoiseParams, build_noise_model, readout_only_noise_model
from qspn.mitigation import AssignmentMatrix, counts_to_vector, mitigate_vector, negative_mass
from qspn.runner import RunConfig, ideal_distribution, run_counts, run_calibration, transpile_for
from qspn.metrics import success_probability as measured_success, shannon_entropy

params = SPNParams(key_bits=4, rounds=2)
config = RunConfig(shots=4096, seed=20260902)
print(params, "| key space:", params.key_space)

## 1. The classical cipher

A 4-bit SPN: `ARK(rk0) → [S-box → P-layer → ARK(rk_r)] × rounds`, using the real
PRESENT S-box. This module is the **ground truth** for everything else — it
imports nothing but the standard library.

In [ ]:
print("PRESENT S-box:", [hex(v)[2:].upper() for v in SBOX])

secret = 0b1101
print(f"\nEncrypting under K = {secret:04b}:")
for pt in range(4):
    ct = encrypt(pt, secret, params)
    assert decrypt(ct, secret, params) == pt
    print(f"  E({pt:04b}) = {ct:04b}   (decrypts back correctly)")

## 2. One pair is not enough

`K ↦ E_K(P)` is not injective for a 4-bit block, so a single known-plaintext
pair usually leaves several consistent keys. This is the same counting argument
that forces AES key-search oracles to encrypt multiple blocks
(Grassl et al. 2016; Jaques et al. 2020).

In [ ]:
print(f"{'pairs':>6} {'worst M':>8} {'mean M':>8} {'unique':>10}")
for n_pairs in (1, 2, 3, 4):
    counts = []
    for K in range(params.key_space):
        pairs = [(pt, encrypt(pt, K, params)) for pt in range(n_pairs)]
        counts.append(len(brute_force_keys(pairs, params)))
    unique = sum(c == 1 for c in counts)
    print(f"{n_pairs:>6} {max(counts):>8} {np.mean(counts):>8.2f} "
          f"{unique:>6}/{params.key_space}")

print(f"\nInformation-theoretic minimum: {required_pairs(params)} pairs")
print("...but the number actually needed is key-dependent. The instance builder")
print("grows the pair count until brute force confirms M = 1:")

pairs = make_attack_instance(secret, params)
print(f"\nK = {secret:04b} needs {len(pairs)} pair(s): {pairs}")
print("consistent keys:", brute_force_keys(pairs, params))

## 3. The reversible circuit *is* the cipher

The load-bearing claim of the project. We check the **exact unitary** against
the classical reference for every key and plaintext — not a sampled outcome.

In [ ]:
from qiskit.quantum_info import Operator

circuit = encryption_circuit(params)
print(f"U_E acts on {circuit.num_qubits} qubits "
      f"({params.key_bits} key + 4 data), ancillas: 0")

unitary = Operator(circuit).data
mismatches = 0
for K in range(params.key_space):
    for pt in range(16):
        column = K + (pt << params.key_bits)
        expected = K + (encrypt(pt, K, params) << params.key_bits)
        if int(np.argmax(np.abs(unitary[:, column]))) != expected:
            mismatches += 1

print(f"checked {params.key_space * 16} (key, plaintext) pairs -> "
      f"{mismatches} mismatches")
assert mismatches == 0

## 4. The oracle, and its uncomputation

`f(K) = 1` iff `E_K(P_i) == C_i` for every pair. Built as
**compute → phase → uncompute**.

Skipping the uncomputation still marks the right states, but leaves the key
register entangled with the data registers, which destroys the interference
Grover needs. So we check both the phases *and* that all probability mass has
returned to the plaintext subspace.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

oracle = key_search_oracle(pairs, params)
print(f"oracle: {oracle.num_qubits} qubits, ops: {dict(oracle.count_ops())}")

prep = QuantumCircuit(oracle.num_qubits)
prep.h(range(params.key_bits))
for reg_index, (pt, _) in enumerate(pairs):
    base = params.key_bits + reg_index * 4
    for bit in range(4):
        if (pt >> bit) & 1:
            prep.x(base + bit)

amplitudes = np.asarray(Statevector.from_instruction(prep).evolve(oracle).data)

offset = 0
for reg_index, (pt, _) in enumerate(pairs):
    offset |= pt << (params.key_bits + reg_index * 4)

norm = 1 / np.sqrt(params.key_space)
marked = set(brute_force_keys(pairs, params))
print("\n key  amplitude   phase   marked?")
for K in range(params.key_space):
    amp = amplitudes[K + offset].real
    print(f" {K:04b}  {amp:+.4f}   {'-1' if amp < 0 else '+1'}     "
          f"{'YES' if K in marked else ''}")

restored = sum(abs(amplitudes[K + offset]) ** 2 for K in range(params.key_space))
print(f"\nprobability mass back in the plaintext subspace: {restored:.12f}")
assert np.isclose(restored, 1.0), "uncomputation is incomplete!"

## 5. Grover amplification — and over-rotation

`k* = ⌊(π/4)√(N/M)⌋`, **not** `√N`. And because Grover is a rotation, doing
*more* than `k*` iterations makes things worse.

In [ ]:
k_star = optimal_iterations(params.key_space, len(marked))
print(f"N = {params.key_space}, M = {len(marked)}  ->  k* = {k_star} "
      f"(sqrt(N) would suggest {int(params.key_space ** 0.5)})")

ks = list(range(k_star + 4))
exact = []
for k in ks:
    probs = ideal_distribution(
        grover_circuit(pairs, k, params, measure=False), params.key_bits, config
    )
    exact.append(measured_success(probs, sorted(marked)))

dense = np.linspace(0, max(ks), 300)
theta = np.arcsin(np.sqrt(len(marked) / params.key_space))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dense, np.sin((2 * dense + 1) * theta) ** 2, "--",
        color="gray", label=r"$\sin^2((2k+1)\theta)$")
ax.plot(ks, exact, "o-", color="#1f4e79", label="simulated")
ax.axvline(k_star, ls="-.", color="#1e8449", label="$k^*$")
ax.axhline(1 / params.key_space, ls=":", color="k", label="random guessing")
ax.set_xlabel("Grover iterations $k$"); ax.set_ylabel("P(correct key)")
ax.legend(frameon=False); ax.grid(alpha=0.3); plt.show()

print(f"P({k_star}) = {exact[k_star]:.4f}   ->   P({k_star + 1}) = "
      f"{exact[k_star + 1]:.4f}   (over-rotation)")

## 6. Noise

Note the transpilation step. A noise model binds to *named basis gates*, so a
circuit still holding composite instructions would run essentially noise-free
while appearing to be noisy — silently invalidating the whole study.

In [ ]:
search = grover_circuit(pairs, k_star, params, measure=True)
transpiled = transpile_for(search, config)
print(f"before transpile: {dict(search.count_ops())}")
print(f"after  transpile: {dict(transpiled.count_ops())}")
print(f"CX = {transpiled.count_ops()['cx']}, depth = {transpiled.depth()}")

noise = NoiseParams()
noisy = counts_to_vector(
    run_counts(transpiled, None, config,
               noise_model=build_noise_model(noise), already_transpiled=True),
    params.key_bits,
)
ideal = ideal_distribution(
    grover_circuit(pairs, k_star, params, measure=False), params.key_bits, config
)

print(f"\nideal P(key) = {measured_success(ideal, sorted(marked)):.4f}")
print(f"noisy P(key) = {measured_success(noisy, sorted(marked)):.4f}")
print(f"output entropy = {shannon_entropy(noisy):.3f} bits "
      f"(uniform = {params.key_bits}.000)")

## 7. Readout-error mitigation, and where it stops working

Characterise `A[i,j] = P(observe i | prepared j)` with `2^n` calibration
circuits, then invert `y = Ax`.

The key comparison: **readout error alone** versus **readout plus gate error**.

In [ ]:
for label, model in (
    ("readout only", readout_only_noise_model(noise)),
    ("readout + gate", build_noise_model(noise)),
):
    cal = run_calibration(params.key_bits, None, config, noise_model=model)
    A = AssignmentMatrix.from_calibration_counts(cal, params.key_bits, config.shots)
    raw = counts_to_vector(
        run_counts(transpiled, None, config, noise_model=model, already_transpiled=True),
        params.key_bits,
    )
    print(f"\n--- {label}  (mean readout fidelity "
          f"{A.mean_readout_fidelity:.3f}, cond(A) = {A.condition_number:.2f})")
    print(f"    raw   P(key) = {measured_success(raw, sorted(marked)):.4f}")
    for method in ("pinv", "clip", "nnls"):
        fixed = mitigate_vector(A.matrix, raw, method)
        valid = np.all(fixed >= -1e-12)
        print(f"    {method:<5s} P(key) = {measured_success(fixed, sorted(marked)):.4f}"
              f"   neg.mass = {negative_mass(fixed):.4f}"
              f"   valid = {'yes' if valid else 'NO'}")

print(f"\nideal P(key) = {measured_success(ideal, sorted(marked)):.4f}")

### What that shows

- **Readout only:** mitigation recovers almost the full ideal signal. `A`
  describes the entire corruption, so inverting it works.
- **Readout + gate error:** mitigation barely moves the result, and both values
  sit at random guessing. Gate error destroyed the state *before* measurement;
  no classical post-processing on the histogram can undo that.
- **`pinv` may report a success probability above the ideal** while placing
  probability mass below zero. That is an unphysical overshoot, not a better
  answer — which is why NNLS is the default.

Run the full study with `qspn-run`, and see `docs/REPORT.md` for the complete
discussion.